[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UPO-Sevilla-Fco-Javier-Lobo-Cabrera/clustering_trait_proteins/blob/main/thioredoxin_mosaic_q.ipynb)

# Testing AlphaFold2 and ESMFold on the Mosaic Q law: human thioredoxin

*Companion notebook to the [Hugging Face article](https://huggingface.co/blog) by Francisco Javier Lobo-Cabrera, Proteins Mosaic Q Project.*

This notebook reproduces the full analysis in a single Colab session. It downloads the experimental structure of human thioredoxin (PDB 1ERT), retrieves the AlphaFold2 prediction from the AlphaFold DB and generates the ESMFold prediction via the ESM Atlas public API, aligns both predictions onto the experimental structure, and computes the Mosaic Q descriptors on the three of them.

Total runtime: about 30 seconds. No GPU required.

**Project links**

- Website: [proteins-mosaic-q.org](https://proteins-mosaic-q.org)
- Hugging Face organization: [ProteinsMosaicQ](https://huggingface.co/ProteinsMosaicQ)
- Python package: [protein-mosaic-q on PyPI](https://pypi.org/project/protein-mosaic-q/)

## 1. Setup

Install the two dependencies we need. `protein-mosaic-q` provides the Q and Q_alt calculations, and pulls in Biopython automatically for structure parsing and alignment.

In [1]:
!pip install protein-mosaic-q --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 20.1 MB/s eta 0:00:00


In [2]:
import urllib.request
import requests
import mosaicq
from Bio.PDB import PDBParser, MMCIFParser, PDBIO, Superimposer

## 2. Download the three structures

We work with three files, all corresponding to the same 105-residue sequence of human thioredoxin (UniProt P10599):

- **Experimental**: PDB 1ERT (X-ray, 1.7 Å resolution), downloaded from RCSB in mmCIF format.
- **AlphaFold2 prediction**: downloaded from the AlphaFold Protein Structure Database.
- **ESMFold prediction**: obtained on the fly by posting the sequence to the ESM Atlas public API.

In [3]:
# Experimental structure from RCSB (mmCIF format)
urllib.request.urlretrieve(
    "https://files.rcsb.org/download/1ERT.cif",
    "1ERT.cif",
)

# AlphaFold2 prediction from the AlphaFold DB (PDB format)
urllib.request.urlretrieve(
    "https://alphafold.ebi.ac.uk/files/AF-P10599-F1-model_v6.pdb",
    "AF2_1ert.pdb",
)

# ESMFold prediction via the ESM Atlas API
sequence = (
    "MVKQIESKTAFQEALDAAGDKLVVVDFSATWCGPCKMIKPFFHSLSEKYSNVIFLEVDVDDCQ"
    "DVASECEVKCMPTFQFFKKGQKVGEFSGANKEKLEATINELV"
)
r = requests.post("https://api.esmatlas.com/foldSequence/v1/pdb/", data=sequence)
with open("esmfold_1ert.pdb", "w") as f:
    f.write(r.text)

print("All three structures downloaded.")

All three structures downloaded.


## 3. Align the predictions onto the experimental structure

The Mosaic Q descriptors are invariant under rotations and translations, so the numerical values of Q and Q_alt do not depend on this alignment. However, if we want to visualize the mosaics side by side (in Jmol, PyMOL or any other viewer), we need the three structures to share the same viewpoint. Otherwise the eye gets distracted by trivial differences in orientation.

We use Biopython's `Superimposer`, which performs a least-squares fit on the Cα atoms and then applies the transformation to all atoms of the mobile structure.

In [4]:
# Parsers: mmCIF for the experimental reference, PDB for the predictions
cif = MMCIFParser(QUIET=True)
pdb = PDBParser(QUIET=True)

# Load the experimental structure once (used as reference for both alignments)
ref = cif.get_structure("exp", "1ERT.cif")
ref_ca = [a for a in ref.get_atoms() if a.get_name() == "CA"]

# Align each prediction onto the experimental structure and save it
for name, pdb_in in [("AF2", "AF2_1ert.pdb"), ("ESMFold", "esmfold_1ert.pdb")]:
    mob = pdb.get_structure(name, pdb_in)
    mob_ca = [a for a in mob.get_atoms() if a.get_name() == "CA"]

    # Least-squares fit on Cα atoms, then apply the transformation to all atoms
    sup = Superimposer()
    sup.set_atoms(ref_ca, mob_ca)
    sup.apply(mob.get_atoms())

    print(f"RMSD {name} vs exp = {sup.rms:.3f} Å")

    # Write the aligned structure to a new file for later visualization
    io = PDBIO()
    io.set_structure(mob)
    io.save(pdb_in.replace(".pdb", "_aligned.pdb"))

RMSD AF2 vs exp = 0.384 Å
RMSD ESMFold vs exp = 0.345 Å


## 4. Compute Q and Q_alt for the three structures

The `mosaicq` package provides two functions that return Q (or Q_alt) together with the number of residues used in the calculation. Both functions accept either PDB or mmCIF files.

Q is built from the pairwise distances between residues of the four main chemical types (hydrophobic, polar, acidic, basic). Q_alt includes an additional "special" group (Cys, Gly, Pro, Sec).

In [5]:
structures = [
    ("Experimental (1ERT)", "1ERT.cif"),
    ("AlphaFold2",          "AF2_1ert.pdb"),
    ("ESMFold",             "esmfold_1ert.pdb"),
]

results = []
for label, path in structures:
    q,     n = mosaicq.calculate_q_and_length(path)
    q_alt, _ = mosaicq.calculate_q_alt_and_length(path)
    results.append((label, n, q, q_alt))
    print(f"{label:22s}  n={n}  Q={q:.4f}  Q_alt={q_alt:.4f}")

Experimental (1ERT)     n=105  Q=19.8076  Q_alt=21.2907
AlphaFold2              n=105  Q=20.0896  Q_alt=21.6161
ESMFold                 n=105  Q=20.1140  Q_alt=21.6253


## 5. Visualize the three mosaics side by side

We render the three aligned structures using the same coloring scheme as the project's community repository:

- **White**: hydrophobic (Ala, Val, Ile, Leu, Met, Phe, Tyr, Trp)
- **Green**: polar (Ser, Thr, Asn, Gln)
- **Orange**: acidic (Asp, Glu)
- **Blue**: basic (Arg, His, Lys)
- **Cyan**: special (Cys, Sec, Gly, Pro)

Residues are drawn as van der Waals spheres. Non-protein atoms (waters, ions) are hidden by only styling protein residues. The equivalent Jmol console script for offline rendering is:

```
select ala or val or ile or leu or met or phe or tyr or trp; color white;
select ser or thr or asn or gln; color green;
select asp or glu; color orange;
select arg or his or lys; color blue;
select cys or sec or gly or pro; color cyan;
select all; spacefill vdw; restrict protein;
```

In [6]:
!pip install py3Dmol --quiet

import py3Dmol
from IPython.display import display, HTML

# Mosaic Q coloring rules: chemical type -> color
MOSAIC_COLORS = [
    (["ALA", "VAL", "ILE", "LEU", "MET", "PHE", "TYR", "TRP"], "white"),   # hydrophobic
    (["SER", "THR", "ASN", "GLN"],                             "green"),   # polar
    (["ASP", "GLU"],                                           "orange"),  # acidic
    (["ARG", "HIS", "LYS"],                                    "blue"),    # basic
    (["CYS", "SEC", "GLY", "PRO"],                             "cyan"),    # special
]

panels = [
    ("1ERT.cif",                 "Experimental (1ERT)"),
    ("AF2_1ert_aligned.pdb",     "AlphaFold2"),
    ("esmfold_1ert_aligned.pdb", "ESMFold"),
]

PANEL_WIDTH  = 240
PANEL_HEIGHT = 260
ZOOM_FACTOR  = 0.9   # ~10% zoom out

def render_panel(path):
    """Render one structure in its own py3Dmol viewer and return the raw HTML."""
    with open(path) as f:
        data = f.read()
    fmt = "cif" if path.endswith(".cif") else "pdb"

    v = py3Dmol.view(width=PANEL_WIDTH, height=PANEL_HEIGHT)
    v.addModel(data, fmt)
    for residues, color in MOSAIC_COLORS:
        v.setStyle({"resn": residues}, {"sphere": {"color": color}})
    v.setBackgroundColor("white")
    v.zoomTo()
    v.zoom(ZOOM_FACTOR)
    return v._make_html()

# Compose a single HTML row with all three viewers and captions
cells_html = ""
for path, label in panels:
    cells_html += (
        f'<div style="width:{PANEL_WIDTH}px; text-align:center;">'
        f'  <div>{render_panel(path)}</div>'
        f'  <div style="font-family:sans-serif; font-size:13px; color:#444; margin-top:6px;">{label}</div>'
        f'</div>'
    )

row_html = f'<div style="display:flex; gap:12px; justify-content:flex-start;">{cells_html}</div>'
display(HTML(row_html))

## 7. Try it with a different protein

The full pipeline above only depends on three inputs: a PDB entry ID, a UniProt accession, and the corresponding sequence. To run the same analysis on another protein, replace these three values and re-run the notebook. A few good candidates for a first exploration:

- **Lysozyme** (PDB 1LYZ, UniProt P00698)
- **Myoglobin** (PDB 1MBN, UniProt P02185)
- **Ubiquitin** (PDB 1UBQ, UniProt P0CG48)

Contributions to the [Proteins Mosaic Q community repository](https://proteins-mosaic-q.org/repository/) are welcome. If you compute the mosaic for a protein that has not been analyzed yet, the [participation guide](https://proteins-mosaic-q.org/participate/) explains how to submit it.

## References

- FAIRsharing.org, Proteins Mosaic Q Project Repository (2026), DOI: 10.25504/FAIRsharing.9f9f9c
- Galaxy Europe, Protein Mosaic Q tool (2026), usegalaxy.eu
- Lobo-Cabrera FJ. "Mosaic Q." Proteopedia, life in 3D. Retrieved July 3, 2026, from https://proteopedia.org/w/Mosaic_Q
- P. J. A. Cock et al., Biopython: freely available Python tools for computational molecular biology and bioinformatics (2009), Bioinformatics 25(11):1422–1423
- Jumper J. et al. (2021). *Highly accurate protein structure prediction with AlphaFold*. Nature 596, 583–589.
- Varadi M. et al. (2024). *AlphaFold Protein Structure Database in 2024*. NAR 52, D368–D375.
- Lin Z. et al. (2023). *Evolutionary-scale prediction of atomic-level protein structure*. Science 379, 1123–1130.
- Weichsel A. et al. (1996). *Crystal structures of reduced, oxidized, and mutated human thioredoxins*. Structure 4, 735–751.